In [1]:
!pip install scikit-learn pandas numpy joblib

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib


In [4]:
np.random.seed(42)

samples = 5000

data = pd.DataFrame({
    "battery_voltage": np.random.normal(28, 4, samples),
    "battery_current": np.random.normal(5, 1.5, samples),
    "temperature": np.random.normal(40, 15, samples),
    "cpu_load": np.random.uniform(10, 100, samples),
    "sensor_error_rate": np.random.uniform(0, 0.2, samples)
})

def label_health(row):
    if row["battery_voltage"] < 20 or row["temperature"] > 80 or row["sensor_error_rate"] > 0.15:
        return 2  # Failure
    elif row["battery_voltage"] < 24 or row["temperature"] > 60:
        return 1  # At Risk
    else:
        return 0  # Healthy

data["target"] = data.apply(label_health, axis=1)

data.head()


,battery_voltage,battery_current,temperature,cpu_load,sensor_error_rate,target
0,29.986857,4.364360,29.822579,45.063572,0.041476,0
1,27.446943,4.319879,35.417508,11.359670,0.081805,0
2,30.590754,2.306535,31.039284,90.844576,0.039101,0
3,34.092119,4.504865,41.656271,57.433960,0.192514,2
4,27.063387,6.099244,57.957678,86.482203,0.030900,0


In [5]:
X = data.drop("target", axis=1)
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [6]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)

print("Logistic Regression Accuracy:", lr_acc)


Logistic Regression Accuracy: 0.806


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print("Random Forest Accuracy:", rf_acc)


Random Forest Accuracy: 0.999


In [8]:
best_model = rf if rf_acc > lr_acc else lr

joblib.dump(best_model, "satellite_health_model.pkl")
print("Model saved as satellite_health_model.pkl")


Model saved as satellite_health_model.pkl
